# 🔬 MulCo-PlantNet — Demo Inference + Grad-CAM

Notebook demo cho mô hình MulCo:
- **Đầu vào**: Ảnh từ tập test + mô tả (caption) tương ứng
- **Đầu ra**: Dự đoán lớp bệnh + heatmap Grad-CAM

Grad-CAM giúp trực quan hoá vùng ảnh mà model "chú ý" nhất khi đưa ra dự đoán.


## 1. Setup — Import và cấu hình


In [ ]:
import os
import sys
import json
import re
import torch
import torch.nn as nn
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image
from torchvision import transforms
from transformers import AutoTokenizer

# Tìm PROJECT_ROOT
current_dir = Path.cwd()
PROJECT_ROOT = current_dir
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

from src.models.mulco import MulCoEndToEnd


## 2. Class mapping và các hàm tiện ích


In [ ]:
# ── Class mapping (28 lớp) ──────────────────────────────────────────────
class_to_idx = {
    'Apple_Scab_Leaf': 0, 'Apple_leaf': 1, 'Apple_rust_leaf': 2, 'Bell_pepper_leaf': 3,
    'Bell_pepper_leaf_spot': 4, 'Blueberry_leaf': 5, 'Cherry_leaf': 6, 'Corn_Gray_leaf_spot': 7,
    'Corn_leaf_blight': 8, 'Corn_rust_leaf': 9, 'Peach_leaf': 10, 'Potato_leaf_early_blight': 11,
    'Potato_leaf_late_blight': 12, 'Raspberry_leaf': 13, 'Soyabean_leaf': 14,
    'Squash_Powdery_mildew_leaf': 15, 'Strawberry_leaf': 16, 'Tomato_Early_blight_leaf': 17,
    'Tomato_Septoria_leaf_spot': 18, 'Tomato_leaf': 19, 'Tomato_leaf_bacterial_spot': 20,
    'Tomato_leaf_late_blight': 21, 'Tomato_leaf_mosaic_virus': 22, 'Tomato_leaf_yellow_virus': 23,
    'Tomato_mold_leaf': 24, 'Tomato_two_spotted_spider_mites_leaf': 25, 'grape_leaf': 26,
    'grape_leaf_black_rot': 27
}
idx_to_class = {v: k for k, v in class_to_idx.items()}

# ── Wrapper để GradCAM chỉ nhận input ảnh (text được fix) ──────────────
class MulCoWrapper(nn.Module):
    """Wrap MulCoEndToEnd để forward chỉ nhận images (fix text tokens)."""
    def __init__(self, model, input_ids, attention_mask):
        super().__init__()
        self.model = model
        self.input_ids = input_ids
        self.attention_mask = attention_mask

    def forward(self, images):
        return self.model(images, self.input_ids, self.attention_mask)

# ── Custom Grad-CAM (không phụ thuộc thư viện bên ngoài) ───────────────
class CustomGradCAM:
    """Grad-CAM đơn giản dùng hook, không cần pytorch-grad-cam."""
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.target_layer.register_forward_hook(self._save_activation)
        self.target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def __call__(self, x, class_idx=None):
        self.model.eval()
        self.model.zero_grad()
        output = self.model(x)

        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        target = output[0, class_idx]
        target.backward()

        gradients = self.gradients.detach().cpu().numpy()[0]   # (C, H, W)
        activations = self.activations.detach().cpu().numpy()[0]  # (C, H, W)

        weights = np.mean(gradients, axis=(1, 2))  # GAP trên spatial → (C,)

        cam = np.zeros(activations.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * activations[i]

        cam = np.maximum(cam, 0)  # ReLU
        cam = cv2.resize(cam, (x.shape[3], x.shape[2]))
        cam = cam - np.min(cam)
        cam = cam / (np.max(cam) + 1e-7)
        return cam, class_idx

# ── Hàm normalize caption (giống lúc training) ─────────────────────────
def normalize_caption(text: str) -> str:
    text = (text or "").strip()
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    for i in range(1, 8):
        text = re.sub(rf"(?i)\bstep\s*{i}\s*:", f"Step {i}:", text)
        text = re.sub(rf"(?i)\bstep{i}\b", f"Step {i}", text)
    text = " ".join(line.strip() for line in text.split("\n") if line.strip())
    text = re.sub(r"\s+", " ", text).strip()
    return text


## 3. Load model và tokenizer


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load model
model = MulCoEndToEnd(num_classes=28).to(device)
ckpt_path = PROJECT_ROOT / "archive/mulco_depth_aug_cb_focal_gem/best_fine_tuned_model.pth"
model.load_state_dict(
    torch.load(ckpt_path, map_location=device, weights_only=True),
    strict=False
)
model.eval()
print(f"✅ Model loaded from: {ckpt_path.name}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
print("✅ Tokenizer loaded (roberta-base)")

# Transform (giống lúc training)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


## 4. Load tập test — ảnh + caption tương ứng


In [ ]:
def load_test_samples(project_root, max_per_class=None):
    """
    Load các mẫu test: ảnh + caption tương ứng.
    
    Cấu trúc test:
    - Ảnh:    data/processed/PlantDocSplited_depth_AUG/test/{class_name}/test_{class_label}_{num}.jpg
    - Caption: data/AIDG/captions_LLaVA/test/{class_name}.json
                Key: {class_name}_{num:05d}.jpg → {"text": "...", "label": ...}
    """
    test_image_root = project_root / "data/processed/PlantDocSplited_depth_AUG/test"
    test_caption_root = project_root / "data/AIDG/captions_LLaVA/test"
    
    samples = []
    
    for class_dir in sorted(test_image_root.iterdir()):
        if not class_dir.is_dir():
            continue
        class_name = class_dir.name
        
        # Load caption JSON cho class này
        caption_file = test_caption_root / f"{class_name}.json"
        if not caption_file.exists():
            print(f"⚠️  Caption file not found: {caption_file.name}, skipping class {class_name}")
            continue
        
        with open(caption_file, "r", encoding="utf-8") as f:
            caption_data = json.load(f)
        
        # Lấy danh sách ảnh test (chỉ ảnh gốc, bỏ depth_suppressed)
        image_files = sorted([
            p for p in class_dir.iterdir()
            if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
            and "_depth_suppressed" not in p.stem
        ])
        
        count = 0
        for img_path in image_files:
            # Tìm caption tương ứng
            # Ảnh: test_Potato leaf late blight_1.jpg
            # Caption key: Potato_leaf_late_blight_00001.jpg
            stem = img_path.stem  # e.g., "test_Potato leaf late blight_1"
            
            # Trích số thứ tự từ tên file
            # Tách prefix "test_" rồi lấy số cuối
            parts = stem.split("_")
            num_str = parts[-1]  # Số cuối cùng, e.g., "1"
            
            try:
                num = int(num_str)
            except ValueError:
                # Nếu không parse được (ảnh augmented), bỏ qua
                continue
            
            # Tạo caption key: {class_name}_{num:05d}.jpg
            caption_key = f"{class_name}_{num:05d}.jpg"
            
            if caption_key in caption_data:
                record = caption_data[caption_key]
                caption_text = normalize_caption(record.get("text", ""))
                
                samples.append({
                    "image_path": img_path,
                    "class_name": class_name,
                    "label": class_to_idx.get(class_name, -1),
                    "caption": caption_text,
                    "caption_key": caption_key,
                })
                count += 1
                if max_per_class and count >= max_per_class:
                    break
            else:
                print(f"  ⚠️  No caption for {img_path.name} (tried key: {caption_key})")
    
    print(f"\n📊 Loaded {len(samples)} test samples across {len(set(s['class_name'] for s in samples))} classes")
    return samples

# Load tất cả test samples
test_samples = load_test_samples(PROJECT_ROOT)


## 5. Hàm inference + Grad-CAM


In [ ]:
def predict_with_gradcam(model, image_path, caption_text, tokenizer, transform,
                         device, target_layer_name="fusion"):
    """
    Dự đoán + tạo heatmap Grad-CAM cho 1 ảnh.
    
    Args:
        target_layer_name: "fusion" (mặc định) hoặc "backbone"
    
    Returns:
        dict với prediction, confidence, top5, gradcam heatmap, v.v.
    """
    # ── Chuẩn bị ảnh ────────────────────────────────────────────────────
    pil_img = Image.open(image_path).convert("RGB")
    input_tensor = transform(pil_img).unsqueeze(0).to(device)
    
    # Ảnh gốc dạng float [0, 1] cho overlay heatmap
    rgb_img = np.array(pil_img.resize((224, 224))).astype(np.float32) / 255.0
    
    # ── Chuẩn bị text ───────────────────────────────────────────────────
    tokens = tokenizer(
        caption_text, padding='max_length', truncation=True,
        max_length=128, return_tensors="pt"
    )
    input_ids = tokens['input_ids'].to(device)
    attention_mask = tokens['attention_mask'].to(device)
    
    # ── Wrap model ──────────────────────────────────────────────────────
    wrapped = MulCoWrapper(model, input_ids, attention_mask)
    
    # Chọn target layer cho Grad-CAM
    if target_layer_name == "backbone":
        target_layer = wrapped.model.image_backbone.norm4
    else:
        # Mặc định: layer cuối của fusion block (sau Restormer)
        target_layer = wrapped.model.fusion_blocks[-1].restormer.ffn.project_out
    
    # ── Chạy Grad-CAM ──────────────────────────────────────────────────
    cam_generator = CustomGradCAM(wrapped, target_layer)
    grayscale_cam, pred_class_idx = cam_generator(input_tensor)
    
    # ── Lấy top-5 predictions ──────────────────────────────────────────
    with torch.no_grad():
        logits = wrapped(input_tensor)
        probs = torch.softmax(logits, dim=1)[0]
    
    top5_probs, top5_indices = probs.topk(5)
    top5 = [
        (idx_to_class[idx.item()], prob.item())
        for idx, prob in zip(top5_indices, top5_probs)
    ]
    
    # ── Tạo heatmap overlay ─────────────────────────────────────────────
    heatmap = cv2.applyColorMap(np.uint8(255 * grayscale_cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    overlay = heatmap + rgb_img
    overlay = overlay / np.max(overlay)
    overlay = np.uint8(255 * overlay)
    
    return {
        "pred_class": idx_to_class[pred_class_idx],
        "pred_idx": pred_class_idx,
        "confidence": probs[pred_class_idx].item(),
        "top5": top5,
        "rgb_img": (rgb_img * 255).astype(np.uint8),
        "heatmap_overlay": overlay,
        "grayscale_cam": grayscale_cam,
    }


## 6. Hàm hiển thị kết quả


In [ ]:
def display_single_result(result, true_class, caption_text, figsize=(14, 5)):
    """Hiển thị kết quả dự đoán + Grad-CAM cho 1 ảnh."""
    is_correct = result["pred_class"] == true_class
    status_icon = "✅" if is_correct else "❌"
    
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1.2])
    
    # ── Ảnh gốc ─────────────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0])
    ax1.imshow(result["rgb_img"])
    ax1.set_title("Original Image", fontsize=12, fontweight='bold')
    ax1.axis('off')
    
    # ── Grad-CAM Heatmap ────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1])
    ax2.imshow(result["heatmap_overlay"])
    ax2.set_title("Grad-CAM Heatmap", fontsize=12, fontweight='bold')
    ax2.axis('off')
    
    # ── Thông tin dự đoán ───────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[2])
    ax3.axis('off')
    
    info_lines = []
    info_lines.append(f"{status_icon}  Prediction: {result['pred_class']}")
    info_lines.append(f"    Confidence: {result['confidence']:.1%}")
    info_lines.append(f"    Ground Truth: {true_class}")
    info_lines.append(f"")
    info_lines.append(f"Top-5 Predictions:")
    for rank, (cls, prob) in enumerate(result["top5"], 1):
        marker = "→" if cls == true_class else " "
        info_lines.append(f"  {marker} {rank}. {cls}: {prob:.1%}")
    
    info_text = "\n".join(info_lines)
    ax3.text(0.05, 0.95, info_text, transform=ax3.transAxes,
             fontsize=10, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.8))
    ax3.set_title("Prediction Info", fontsize=12, fontweight='bold')
    
    # Caption (hiển thị rút gọn)
    short_caption = caption_text[:120] + "..." if len(caption_text) > 120 else caption_text
    fig.suptitle(f"Caption: \"{short_caption}\"", fontsize=9, style='italic', color='gray', y=0.02)
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.08)
    plt.show()


def display_grid_results(results_list, ncols=4, figsize_per_cell=(3.5, 4.5)):
    """Hiển thị lưới kết quả (ảnh gốc + heatmap overlay + nhãn)."""
    n = len(results_list)
    nrows = (n + ncols - 1) // ncols
    fig_w = figsize_per_cell[0] * ncols
    fig_h = figsize_per_cell[1] * nrows

    fig, axes = plt.subplots(nrows * 2, ncols, figsize=(fig_w, fig_h))
    if nrows * 2 == 1:
        axes = axes[np.newaxis, :]
    if ncols == 1:
        axes = axes[:, np.newaxis]
    
    for i, item in enumerate(results_list):
        row = (i // ncols) * 2
        col = i % ncols
        result = item["result"]
        true_class = item["true_class"]
        is_correct = result["pred_class"] == true_class
        
        # Ảnh gốc
        axes[row, col].imshow(result["rgb_img"])
        axes[row, col].axis('off')
        color = "green" if is_correct else "red"
        icon = "✓" if is_correct else "✗"
        axes[row, col].set_title(
            f"GT: {true_class.replace('_', ' ')}",
            fontsize=8, color='black'
        )
        
        # Heatmap
        axes[row + 1, col].imshow(result["heatmap_overlay"])
        axes[row + 1, col].axis('off')
        axes[row + 1, col].set_title(
            f"{icon} Pred: {result['pred_class'].replace('_', ' ')}\n({result['confidence']:.0%})",
            fontsize=8, fontweight='bold', color=color
        )
    
    # Ẩn các ô trống
    for i in range(n, nrows * ncols):
        row = (i // ncols) * 2
        col = i % ncols
        axes[row, col].axis('off')
        axes[row + 1, col].axis('off')
    
    plt.suptitle("MulCo-PlantNet — Test Set Predictions + Grad-CAM", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


## 7. Demo: Dự đoán từng ảnh chi tiết

Chọn một vài ảnh test cụ thể để xem kết quả chi tiết.


In [ ]:
# Chọn 1 ảnh từ mỗi class để demo (hoặc tuỳ chỉnh)
import random
random.seed(42)

# Lấy 1 ảnh ngẫu nhiên từ mỗi class
classes_seen = set()
demo_samples = []
shuffled = list(test_samples)
random.shuffle(shuffled)
for s in shuffled:
    if s["class_name"] not in classes_seen:
        demo_samples.append(s)
        classes_seen.add(s["class_name"])

demo_samples = sorted(demo_samples, key=lambda x: x["class_name"])
print(f"Demo: {len(demo_samples)} ảnh (1 per class)")


In [ ]:
# Chạy inference + Grad-CAM cho từng ảnh demo
for i, sample in enumerate(demo_samples[:5]):  # Hiển thị 5 ảnh đầu tiên
    print(f"\n{'='*60}")
    print(f"[{i+1}] {sample['image_path'].name}")
    print(f"    Class: {sample['class_name']}")
    
    result = predict_with_gradcam(
        model, sample["image_path"], sample["caption"],
        tokenizer, transform, device
    )
    
    display_single_result(result, sample["class_name"], sample["caption"])


## 8. Demo: Lưới tổng quan toàn bộ tập test

Chạy inference trên **tất cả** ảnh test và hiển thị dưới dạng lưới.


In [ ]:
# Chạy inference trên tất cả test samples
all_results = []
correct = 0
total = 0

print("🔄 Running inference on all test samples...")
for i, sample in enumerate(test_samples):
    result = predict_with_gradcam(
        model, sample["image_path"], sample["caption"],
        tokenizer, transform, device
    )
    
    is_correct = result["pred_class"] == sample["class_name"]
    if is_correct:
        correct += 1
    total += 1
    
    all_results.append({
        "result": result,
        "true_class": sample["class_name"],
        "caption": sample["caption"],
        "image_name": sample["image_path"].name,
    })
    
    if (i + 1) % 20 == 0:
        print(f"  Processed {i+1}/{len(test_samples)} samples...")

print(f"\n{'='*60}")
print(f"📊 Overall Accuracy: {correct}/{total} = {correct/total:.1%}")


In [ ]:
# Hiển thị lưới kết quả (lấy tối đa 28 ảnh, 1 per class)
grid_items = []
classes_shown = set()
for item in all_results:
    if item["true_class"] not in classes_shown:
        grid_items.append(item)
        classes_shown.add(item["true_class"])

display_grid_results(grid_items, ncols=4)


## 9. Phân tích: Các trường hợp dự đoán sai


In [ ]:
wrong_predictions = [item for item in all_results if item["result"]["pred_class"] != item["true_class"]]

if wrong_predictions:
    print(f"❌ Có {len(wrong_predictions)} trường hợp dự đoán sai:\n")
    for item in wrong_predictions:
        r = item["result"]
        print(f"  • {item['image_name']}")
        print(f"    GT: {item['true_class']}  →  Pred: {r['pred_class']} ({r['confidence']:.1%})")
        print()
    
    # Hiển thị chi tiết từng trường hợp sai
    for item in wrong_predictions[:10]:  # Tối đa 10 trường hợp
        display_single_result(item["result"], item["true_class"], item["caption"])
else:
    print("✅ Tất cả dự đoán đều đúng!")
